# v9 Post-2015 Breadth PR & Win-Rate Validation

正式 Drive root 已確認為 `/content/drive/MyDrive/Quant_Research/taiwan-market-breadth-research`。Notebook 只驗證既有資料夾，不會自動建立或猜測路徑。

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, tempfile
import warnings
print('Python:', sys.version)
print('Python executable:', sys.executable)
if sys.version_info[:2] != (3, 11):
    warnings.warn(f'Repository baseline is Python 3.11; validating runtime {sys.version_info.major}.{sys.version_info.minor} before continuing.')
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/Quant_Research/taiwan-market-breadth-research')
print('DRIVE_OUTPUT_ROOT:', DRIVE_OUTPUT_ROOT)
print('exists:', DRIVE_OUTPUT_ROOT.exists())
print('is_dir:', DRIVE_OUTPUT_ROOT.is_dir())
if not DRIVE_OUTPUT_ROOT.exists() or not DRIVE_OUTPUT_ROOT.is_dir():
    raise FileNotFoundError('Expected existing Drive folder not found: /content/drive/MyDrive/Quant_Research/taiwan-market-breadth-research')

In [ ]:
REPO_URL = 'https://github.com/hh4832/taiwan-market-breadth-research.git'
REPO_DIR = Path('/content/taiwan-market-breadth-research')
BRANCH = 'feature/v9-post2015-breadth-pr-winrate'
os.chdir('/content')  # never delete the active cwd
if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise RuntimeError(f'Refusing to remove non-git path: {REPO_DIR}')
    remote = subprocess.run(['git','-C',str(REPO_DIR),'remote','get-url','origin'], check=True, capture_output=True, text=True).stdout.strip()
    if 'hh4832/taiwan-market-breadth-research' not in remote:
        raise RuntimeError(f'Refusing to remove unexpected repository: {remote}')
    shutil.rmtree(REPO_DIR)
env = os.environ.copy(); env['GIT_TERMINAL_PROMPT'] = '0'
token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
askpass_dir = None
if token:
    askpass_dir = Path(tempfile.mkdtemp(prefix='git-askpass-'))
    askpass = askpass_dir / 'askpass.sh'
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700); env.update({'GIT_ASKPASS': str(askpass), 'GITHUB_TOKEN': token})
try:
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)], check=True, env=env)
except subprocess.CalledProcessError as exc:
    raise RuntimeError('Clone failed. If the repository is private, add GITHUB_TOKEN to Colab Secrets with repository read access.') from exc
finally:
    if askpass_dir: shutil.rmtree(askpass_dir, ignore_errors=True)
subprocess.run(['git','-C',str(REPO_DIR),'remote','set-url','origin',REPO_URL], check=True)
os.chdir(REPO_DIR)
assert Path.cwd().name == 'taiwan-market-breadth-research'
remote = subprocess.run(['git','remote','get-url','origin'], check=True, capture_output=True, text=True).stdout.strip()
assert remote == REPO_URL
for args in (['pwd'], ['git','remote','-v'], ['git','branch','--show-current'], ['git','status','--short'], ['git','rev-parse','HEAD']):
    subprocess.run(args, check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True, env={**os.environ, 'PYTHONPATH': str(REPO_DIR/'src')})
for package in ('finlab','pandas','numpy','scipy','statsmodels','pyarrow','openpyxl'):
    module = __import__(package)
    print(package, getattr(module, '__version__', 'version unavailable'))
try:
    import finlab
    try:
        from google.colab import userdata
        finlab_token = userdata.get('FINLAB_API_TOKEN')
    except Exception:
        finlab_token = None
    if finlab_token:
        finlab.login(finlab_token)
    else:
        print('FINLAB_API_TOKEN not found; using the current FinLab login/session if available.')
except Exception as exc:
    raise RuntimeError('FinLab authentication failed. Configure FINLAB_API_TOKEN or complete FinLab login.') from exc

In [ ]:
sys.path.insert(0, str(REPO_DIR / 'src'))
from market_breadth.config import V9Config
from market_breadth.pipeline_v9 import run_v9
from market_breadth.run_context import archive_run, create_run_context
config = V9Config(cache_dir=REPO_DIR/'cache', output_dir=REPO_DIR/'output')
run_context = create_run_context(config.version_slug, config.output_dir)
planned_drive = DRIVE_OUTPUT_ROOT / run_context.run_id
print('Planned local run:', run_context.local_run_dir)
print('Planned Drive archive:', planned_drive)
assert not planned_drive.exists(), f'Archive already exists: {planned_drive}'
RESULTS = run_v9(config, run_context, DRIVE_OUTPUT_ROOT)

In [ ]:
required = ['market_breadth_summary.xlsx','daily_dataset.parquet','run_info.txt','validation_summary.md','plots']
for name in required:
    path = run_context.local_run_dir / name
    assert path.exists(), f'Missing local output: {path}'
    if path.is_file(): assert path.stat().st_size > 0
assert str(DRIVE_OUTPUT_ROOT) == '/content/drive/MyDrive/Quant_Research/taiwan-market-breadth-research'
drive_run_dir = archive_run(run_context.local_run_dir, DRIVE_OUTPUT_ROOT, run_context.run_id, require_official_root=True)
assert drive_run_dir.parent.resolve() == DRIVE_OUTPUT_ROOT.resolve()
canonical = RESULTS['results'].loc[~RESULTS['results'].is_duplicate_hypothesis]
print('Study version:', config.study_version)
print('Run ID:', run_context.run_id)
print('Git commit:', run_context.git_commit)
print('Branch:', run_context.git_branch)
print('Python:', sys.version)
print('Actual sample:', RESULTS['dataset'].index.min(), 'to', RESULTS['dataset'].index.max())
print('Local output:', run_context.local_run_dir)
print('Drive output root:', DRIVE_OUTPUT_ROOT)
print('Drive run dir:', drive_run_dir)
print('Summary Excel:', RESULTS['output_paths']['summary'])
print('Dataset:', RESULTS['output_paths']['dataset'])
print('Validation summary:', RESULTS['output_paths']['validation_summary'])
print('Number of canonical hypotheses:', len(canonical))
print('Mean-return global FDR significant:', canonical.mean_return_FDR_global.lt(.05).sum())
print('Win-rate global FDR significant:', canonical.win_rate_FDR_global.lt(.05).sum())
print('Archive validation: PASS')